<a href="https://colab.research.google.com/github/rahafalnjjar49-cloud/Robot-simulation/blob/main/Robotic_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from enum import Enum
import time

In [ ]:
class CellType(Enum):
    EMPTY = 0
    OBSTACLE = 1
    GOAL = 2
    PATH = 3

class Direction(Enum):
    NORTH = 0
    EAST = 1
    SOUTH = 2
    WEST = 3

class Environment:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = [[CellType.EMPTY for _ in range(width)] for _ in range(height)]

    def is_obstacle(self, x, y):
        if not (0 <= x < self.width and 0 <= y < self.height): return True
        return self.grid[y][x] == CellType.OBSTACLE

    def display(self, robot):
        icons = {Direction.NORTH: "^", Direction.EAST: ">", Direction.SOUTH: "v", Direction.WEST: "<"}
        print("\n" + "="*25)
        for y in range(self.height):
            for x in range(self.width):
                if robot.x == x and robot.y == y:
                    print(f"\033[92m{icons[robot.direction]}\033[0m", end=" ") # Green Robot
                elif self.grid[y][x] == CellType.OBSTACLE:
                    print("\033[91m#\033[0m", end=" ") # Red Obstacle
                elif self.grid[y][x] == CellType.GOAL:
                    print("\033[94mG\033[0m", end=" ") # Blue Goal
                elif self.grid[y][x] == CellType.PATH:
                    print("\033[90m.\033[0m", end=" ") # Grey Path
                else:
                    print(" ", end=" ")
            print()
        print("="*25)

class Robot:
    def __init__(self, x, y, goal_x, goal_y):
        self.x, self.y = x, y
        self.goal_x, self.goal_y = goal_x, goal_y
        self.direction = Direction.EAST

    def move(self, env):
        env.grid[self.y][self.x] = CellType.PATH

        dx = self.goal_x - self.x
        dy = self.goal_y - self.y

        if abs(dx) >= abs(dy) and dx != 0:
            target_dir = Direction.EAST if dx > 0 else Direction.WEST
        else:
            target_dir = Direction.SOUTH if dy > 0 else Direction.NORTH

        nx, ny = self.get_next_pos(target_dir)
        if not env.is_obstacle(nx, ny):

            self.direction = target_dir
            self.x, self.y = nx, ny
            return f"Moving towards goal: {target_dir.name}"
        else:
            print(f"\033[93m[!] OBSTACLE at ({nx},{ny})! Changing path...\033[0m")
            for rotation in [1, -1, 2]:
                alt_dir = Direction((target_dir.value + rotation) % 4)
                ax, ay = self.get_next_pos(alt_dir)
                if not env.is_obstacle(ax, ay):
                    print(f"\033[92m[+] Found clear path: Turning {alt_dir.name}\033[0m")
                    self.direction = alt_dir
                    self.x, self.y = ax, ay
                    return f"Avoided obstacle by turning {alt_dir.name}"
            return "STUCK"

    def get_next_pos(self, direction):
        moves = {Direction.NORTH: (0, -1), Direction.SOUTH: (0, 1),
                 Direction.EAST: (1, 0), Direction.WEST: (-1, 0)}
        dx, dy = moves[direction]
        return self.x + dx, self.y + dy

def main():
    env = Environment(12, 8)
    for y in range(1, 7): env.grid[y][6] = CellType.OBSTACLE

    goal = (10, 4)
    env.grid[goal[1]][goal[0]] = CellType.GOAL

    robot = Robot(1, 4, goal[0], goal[1])

    print("Starting Simulation: Watch the robot change its path!")
    for step in range(30):
        env.display(robot)
        if (robot.x, robot.y) == goal:
            print("\033[94m*** SUCCESS: GOAL REACHED! ***\033[0m")
            break

        print(f"Step {step}: Position ({robot.x}, {robot.y})")
        status = robot.move(env)
        if status == "STUCK":
            print("Robot is stuck!")
            break
        time.sleep(0.1)

if __name__ == "__main__":
    main()

Starting Simulation: Watch the robot change its path!

                        
            #           
            #           
            #           
  >         #       G   
            #           
            #           
                        
Step 0: Position (1, 4)

                        
            #           
            #           
            #           
  . >       #       G   
            #           
            #           
                        
Step 1: Position (2, 4)

                        
            #           
            #           
            #           
  . . >     #       G   
            #           
            #           
                        
Step 2: Position (3, 4)

                        
            #           
            #           
            #           
  . . . >   #       G   
            #           
            #           
                        
Step 3: Position (4, 4)

                        
            #       